In [9]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


In [10]:


# =========================
# Statistics
# =========================
def concordance_correlation_coefficient(x, y):
    """
    Calculate Lin's concordance correlation coefficient.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Retain only paired finite values.
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    n = len(x)
    if n < 2:
        return np.nan

    mx, my = x.mean(), y.mean()
    vx, vy = x.var(ddof=1), y.var(ddof=1)

    if vx == 0 and vy == 0:
        return np.nan

    cov = np.cov(x, y, ddof=1)[0, 1]

    ccc = (
        2 * cov
        / (vx + vy + (mx - my) ** 2)
    )

    return ccc


def compute_stats(x, y):
    """
    Calculate Pearson's r, Pearson p value, CCC, and degrees of freedom.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Retain only paired finite values.
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    n = len(x)

    if n < 2:
        return np.nan, np.nan, np.nan, np.nan

    r, p = stats.pearsonr(x, y)
    ccc = concordance_correlation_coefficient(x, y)
    df = n - 2

    return r, p, ccc, df


# =========================
# Axis-square helper
# =========================
def _apply_axis_square_same_scale(ax, x, y):
    """
    Apply identical x-axis and y-axis ranges and an equal aspect ratio.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if x.size == 0 or y.size == 0:
        return

    # Determine the common range from observed and predicted values.
    mn = min(x.min(), y.min())
    mx = max(x.max(), y.max())

    # Add a small margin around the data.
    pad = 0.03 * (mx - mn) if mx > mn else 0.5

    mn -= pad
    mx += pad

    ax.set_xlim(mn, mx)
    ax.set_ylim(mn, mx)

    # Use the same physical scale for the two axes.
    ax.set_aspect(
        "equal",
        adjustable="box"
    )


# =========================
# Read data + strict log10
# =========================
def read_xy_and_log10_by_cols(
    file_path,
    sheet_name,
    x_col,
    y_col
):
    """
    Read selected observed and predicted columns, perform strict log10
    transformation, and remove NaN and infinite values.
    """
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=0
    )

    # Remove possible leading or trailing spaces in column names.
    df.columns = df.columns.astype(str).str.strip()

    if x_col not in df.columns or y_col not in df.columns:
        raise ValueError(
            f"[{sheet_name}] Missing columns. "
            f"Need '{x_col}' and '{y_col}'. "
            f"Found: {list(df.columns)}"
        )

    x_raw = pd.to_numeric(
        df[x_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_raw = pd.to_numeric(
        df[y_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    # Strict log10 transformation.
    # Zero and negative values become non-finite and are removed below.
    x = np.log10(x_raw)
    y = np.log10(y_raw)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    df_xy = pd.DataFrame({
        "Real": x,
        "Predict": y
    })

    return df_xy


# =========================
# Draw jointplot
# =========================
def draw_joint_with_marginals(
    df_xy,
    name_for_save,
    out_dir
):
    """
    Draw observed-versus-predicted values with linear regression and
    marginal distributions. Statistics are calculated on the same
    log10-transformed data used in the plot.
    """
    sns.set_style(
        "white",
        rc={
            "xtick.bottom": True,
            "ytick.left": True,
            "xtick.major.size": 4,
            "ytick.major.size": 4,
            "xtick.major.width": 1.0,
            "ytick.major.width": 1.0,
            "xtick.direction": "out",
            "ytick.direction": "out",
        }
    )

    sns.set_palette(
        sns.color_palette("deep")
    )

    fig = sns.jointplot(
        data=df_xy,
        x="Real",
        y="Predict",
        kind="reg"
    )

    fig.ax_joint.set_xlabel(
        "Log(Pathologies)",
        fontweight="bold",
        fontsize=14
    )

    fig.ax_joint.set_ylabel(
        "Log(Predicted)",
        fontweight="bold",
        fontsize=14
    )

    # Use identical x-axis and y-axis scales.
    _apply_axis_square_same_scale(
        fig.ax_joint,
        df_xy["Real"],
        df_xy["Predict"]
    )

    # Add marginal histograms and KDE curves.
    fig.plot_marginals(
        sns.histplot,
        kde=True
    )

    # Calculate statistics using the plotted log10-transformed data.
    r, p, ccc, df_ = compute_stats(
        df_xy["Real"],
        df_xy["Predict"]
    )

    stat_txt = (
        f"R = {r:.3f}\n"
        f"p = {p:.2e}\n"
        f"df = {df_:d}\n"
        f"CCC = {ccc:.3f}"
    )

    fig.ax_joint.text(
        0.04,
        0.96,
        stat_txt,
        transform=fig.ax_joint.transAxes,
        ha="left",
        va="top",
        fontsize=11,
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="none",
            alpha=0.85
        )
    )

    # Force visible outward ticks on the bottom and left axes.
    fig.ax_joint.tick_params(
        axis="both",
        which="both",
        direction="out",
        bottom=True,
        left=True,
        length=4,
        width=1.0
    )

    fig.ax_joint.xaxis.set_ticks_position("bottom")
    fig.ax_joint.yaxis.set_ticks_position("left")

    # Create the gene-specific output folder.
    out_dir = Path(out_dir)
    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    save_pdf = (
        out_dir
        / f"{name_for_save}_joint_with_marginals.pdf"
    )

    save_tif = (
        out_dir
        / f"{name_for_save}_joint_with_marginals.tif"
    )

    fig.savefig(
        save_pdf,
        format="pdf"
    )

    fig.savefig(
        save_tif,
        format="tif",
        dpi=300,
        facecolor="white",
        transparent=False
    )

    plt.close(fig.fig)

    print("Saved:")
    print(save_pdf)
    print(save_tif)



In [11]:

# =========================
# Project paths
# =========================
# The Jupyter Notebook working directory should be:
# Code/Figures Python
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "Data"
RESULTS_DIR = PROJECT_ROOT / "Results"


# =========================
# Input and output
# =========================
xlsx_path = (
    DATA_DIR
    / "Gene_Model_Fit_Results.xlsx"
)

OUT_ROOT = (
    RESULTS_DIR
    / "Gene_Model_Fit_Results"
)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)



In [12]:

# ============================================================
# Run: MSA best gene Kctd4
# ============================================================
sheet_msa_kctd4 = "MSA_Best_Gene_Kctd4"

out_msa_kctd4 = (
    OUT_ROOT
    / "MSA_Best_Gene_Kctd4"
)

df_msa_kctd4_3 = read_xy_and_log10_by_cols(
    xlsx_path,
    sheet_msa_kctd4,
    "Real3",
    "Predict3"
)

draw_joint_with_marginals(
    df_msa_kctd4_3,
    name_for_save="MSA_Best_Gene_Kctd4_Real3_Predict3",
    out_dir=out_msa_kctd4
)


# ============================================================
# Run: PFF best gene Dok5
# ============================================================
sheet_pff_dok5 = "PFF_Best_Gene_Dok5"

out_pff_dok5 = (
    OUT_ROOT
    / "PFF_Best_Gene_Dok5"
)


# PFF Dok5: 3 MPI
df_pff_dok5_3 = read_xy_and_log10_by_cols(
    xlsx_path,
    sheet_pff_dok5,
    "Real3",
    "Predict3"
)

draw_joint_with_marginals(
    df_pff_dok5_3,
    name_for_save="PFF_Best_Gene_Dok5_Real3_Predict3",
    out_dir=out_pff_dok5
)


# PFF Dok5: 6 MPI
df_pff_dok5_6 = read_xy_and_log10_by_cols(
    xlsx_path,
    sheet_pff_dok5,
    "Real6",
    "Predict6"
)

draw_joint_with_marginals(
    df_pff_dok5_6,
    name_for_save="PFF_Best_Gene_Dok5_Real6_Predict6",
    out_dir=out_pff_dok5
)


# ============================================================
# Run: MSA validated gene Ube2g2
# ============================================================
sheet_msa_ube2g2 = "MSA_Validated_Gene_Ube2g2"

out_msa_ube2g2 = (
    OUT_ROOT
    / "MSA_Validated_Gene_Ube2g2"
)

df_msa_ube2g2_3 = read_xy_and_log10_by_cols(
    xlsx_path,
    sheet_msa_ube2g2,
    "Real3",
    "Predict3"
)

draw_joint_with_marginals(
    df_msa_ube2g2_3,
    name_for_save="MSA_Validated_Gene_Ube2g2_Real3_Predict3",
    out_dir=out_msa_ube2g2
)


# ============================================================
# Run: MSA validated gene Dlg2
# ============================================================
sheet_msa_dlg2 = "MSA_Validated_Gene_Dlg2"

out_msa_dlg2 = (
    OUT_ROOT
    / "MSA_Validated_Gene_Dlg2"
)

df_msa_dlg2_3 = read_xy_and_log10_by_cols(
    xlsx_path,
    sheet_msa_dlg2,
    "Real3",
    "Predict3"
)

draw_joint_with_marginals(
    df_msa_dlg2_3,
    name_for_save="MSA_Validated_Gene_Dlg2_Real3_Predict3",
    out_dir=out_msa_dlg2
)


# ============================================================
# Run: MSA validated gene Gas7
# ============================================================
sheet_msa_gas7 = "MSA_Validated_Gene_Gas7"

out_msa_gas7 = (
    OUT_ROOT
    / "MSA_Validated_Gene_Gas7"
)

df_msa_gas7_3 = read_xy_and_log10_by_cols(
    xlsx_path,
    sheet_msa_gas7,
    "Real3",
    "Predict3"
)

draw_joint_with_marginals(
    df_msa_gas7_3,
    name_for_save="MSA_Validated_Gene_Gas7_Real3_Predict3",
    out_dir=out_msa_gas7
)

C:\Users\forge\AppData\Local\Temp\ipykernel_32868\3026565486.py:135: RuntimeWarning: divide by zero encountered in log10
  x = np.log10(x_raw)
C:\Users\forge\AppData\Local\Temp\ipykernel_32868\3026565486.py:136: RuntimeWarning: divide by zero encountered in log10
  y = np.log10(y_raw)


Saved:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\MSA_Best_Gene_Kctd4\MSA_Best_Gene_Kctd4_Real3_Predict3_joint_with_marginals.pdf
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\MSA_Best_Gene_Kctd4\MSA_Best_Gene_Kctd4_Real3_Predict3_joint_with_marginals.tif


C:\Users\forge\AppData\Local\Temp\ipykernel_32868\3026565486.py:135: RuntimeWarning: divide by zero encountered in log10
  x = np.log10(x_raw)
C:\Users\forge\AppData\Local\Temp\ipykernel_32868\3026565486.py:136: RuntimeWarning: divide by zero encountered in log10
  y = np.log10(y_raw)


Saved:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\PFF_Best_Gene_Dok5\PFF_Best_Gene_Dok5_Real3_Predict3_joint_with_marginals.pdf
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\PFF_Best_Gene_Dok5\PFF_Best_Gene_Dok5_Real3_Predict3_joint_with_marginals.tif


C:\Users\forge\AppData\Local\Temp\ipykernel_32868\3026565486.py:136: RuntimeWarning: divide by zero encountered in log10
  y = np.log10(y_raw)


...

Saved:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\MSA_Validated_Gene_Ube2g2\MSA_Validated_Gene_Ube2g2_Real3_Predict3_joint_with_marginals.pdf
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\MSA_Validated_Gene_Ube2g2\MSA_Validated_Gene_Ube2g2_Real3_Predict3_joint_with_marginals.tif


C:\Users\forge\AppData\Local\Temp\ipykernel_32868\3026565486.py:135: RuntimeWarning: divide by zero encountered in log10
  x = np.log10(x_raw)
C:\Users\forge\AppData\Local\Temp\ipykernel_32868\3026565486.py:136: RuntimeWarning: divide by zero encountered in log10
  y = np.log10(y_raw)


Saved:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\MSA_Validated_Gene_Dlg2\MSA_Validated_Gene_Dlg2_Real3_Predict3_joint_with_marginals.pdf
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\MSA_Validated_Gene_Dlg2\MSA_Validated_Gene_Dlg2_Real3_Predict3_joint_with_marginals.tif


C:\Users\forge\AppData\Local\Temp\ipykernel_32868\3026565486.py:135: RuntimeWarning: divide by zero encountered in log10
  x = np.log10(x_raw)
C:\Users\forge\AppData\Local\Temp\ipykernel_32868\3026565486.py:136: RuntimeWarning: divide by zero encountered in log10
  y = np.log10(y_raw)


Saved:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\MSA_Validated_Gene_Gas7\MSA_Validated_Gene_Gas7_Real3_Predict3_joint_with_marginals.pdf
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\Gene_Model_Fit_Results\MSA_Validated_Gene_Gas7\MSA_Validated_Gene_Gas7_Real3_Predict3_joint_with_marginals.tif
